Name

Labpartner(s)

In [ ]:
#import statements go here
import matplotlib.pyplot as plt
import numpy as np

# Class 6.1

Today we will start with fiunction sharing and do more matplotlib.

# Warmups 6.1

**W.1** Write a function that loops through a list of numbers and checks if they are odd or even, then returns a new list of "odd" or "even" for each element in the input list. For example:

Given the list [3,8,7] the function would return
["odd", "even", "odd"]

# Lecture 6.1

### Agenda:

- Show us your functions
- Questions
- xarray package and plotting netcdf files


### Show us your functions (from Lab 5.2) - first 4 people today, the rest on Wed

### Questions

### Loading and plotting netcdf files using xarray

Most modeling data output is in the form of netcdf files, as they can store more data (in binary) using less memory. Netcdf files are great because they tell you all about what is in the file (the variables and their units) with their metadata, which is kind of like the docstring we made for our function. There are a number of command line (unix-based) utilities for dealing with netcdf files, which I am not planning to cover in this course (though I use these all the time). Hit me up if you want some tutorials on this, or if enough of you are keen I will put some unix tutorials in the schedule.

Xarray is a python package that does analysis and basic plotting of netcdf files. This is actively being developed by folks like the pangeo consortium (https://pangeo.io), which is creating a number of python utilities for big data geoscience, like dealing with massive amounts of climate model output. There are other packages that can be used for parsing netcdf files, but they are cumbersome and clunky. Trust me, xarray is the best thing since sliced bread for big data geoscience. 

Let's grab some data and start playing with it. We are going to use the HYCOM Gulf of Mexico Analysis output, which is basically weather prediction for our local ocean made by the Navy, freely available. https://www.hycom.org.

In [ ]:
import xarray as xr
# make sure you also have nectdf4 installed!

### Part 1: Load in the file (always the hardest part)!

We want the HYCOM GoM reanalysis product: https://www.hycom.org/dataserver/gom/gom-reanalysis
And we are going to use the coarser resolution (1/25 degree) version for computational speed, the "HYCOM-TSIS 1/25º Gulf of Mexico Reanalysis (GOMe0.04)" version, which goes from 2024-09-01 to present.

There are two ways to get this data, we can click around and download it, or we can use the opendap link (see http://xarray.pydata.org/en/stable/io.html).

In [ ]:
# here I am going to grab the hindcast they made for Jan 1 2026. 

#First let's navigate to the website and download the 2d file.

# Note I can use the download the file I got by clicking on the https server link and putting the correct path to load in:
file_path= "/Users/chsharri/Downloads/039_archv.2026_001_00_2d.nc" # you will have to change this for your computer

# now upload from the file I downloaded manually:
hycom_data = xr.open_dataset(file_path, decode_times=False)

# honestly I don't know why you need the decode_times bit with open_dataset
# I just know it doesn't work most of the time if you leave it out (kudos for anyone who figures it out!)


In [ ]:
hycom_data

The result is an xarray dataset, which is similar to the pandas dataframes you have been using. It has dimensions, coordinates and variables. The first thing to do when you get a dataset is to figure out what is in it and explore it a bit.

In [ ]:
# what is the lat spacing and domain?

hycom_data.Latitude

# looks like it goes from 18.09 N to 31.96 N and the spacing is 0.04 degrees, i.e. 1/25, so that checks
# 1 degree is about 100 km, so thats 4 km model resolution

In [ ]:
hycom_data.Longitude

In [ ]:
hycom_data.ssh

We will plot this in the next section

In [ ]:
# alternatley, with a little help from ChatGPT I can make a nice workflow to get the file from the opendap (thredds) server

# Paste the URL copied from the HYCOM data website
file_url = (
    "https://data.hycom.org/datasets/GOMe0.04/expt_03.9/data/archive/2026/"
    "039_archv.2026_001_01_2d.nc"
)

# Convert the download URL to an OPeNDAP URL
opendap_url = file_url.replace(
    "https://data.hycom.org/",
    "https://tds.hycom.org/thredds/dodsC/"
)

print(opendap_url)

# Open the remote NetCDF dataset
hycom_data2 = xr.open_dataset(
    opendap_url,
    engine="netcdf4",
    decode_times=False
)

hycom_data2

# note, this would make a great function

However, what I really want is to plot SST, so I need a file with temperature, let's check out the other file type

In [ ]:
# modify the file to get the 3D output file

# Paste the URL copied from the HYCOM data website
file_url = (
    "https://data.hycom.org/datasets/GOMe0.04/expt_03.9/data/archive/2026/"
    "039_archv.2026_001_01_3z.nc"
)

# Convert the download URL to an OPeNDAP URL
opendap_url = file_url.replace(
    "https://data.hycom.org/",
    "https://tds.hycom.org/thredds/dodsC/"
)

print(opendap_url)

# Open the remote NetCDF dataset
hycom_data_3D = xr.open_dataset(
    opendap_url,
    engine="netcdf4",
    decode_times=False
)

hycom_data_3D

### Part2: Basic plotting with xarray (not publication ready!)

Note that one of the amazing things about xarray, is that it actually does not go and get the data until you call for it, so this will take a minute to upload.

In [ ]:
hycom_data.ssh.plot()

Note that xarray, like pandas, uses matplotlib for plotting, and that it figured out to use the blue to red colormap based on the type of data. Pretty cool. 

Let's plot some temperature data and see how it compares. Since temperature data is given for the whole depth, we have to select a level.

water_temp
(MT, Depth, Latitude, Longitude)

In [ ]:
hycom_data_3D.water_temp[0,0,:,:]

Those plots are really small. I like to change the default matplotlib preferences to make my plots bigger.

In [ ]:
# change all the defaults (usually I stick this up with the import statements)

import matplotlib as mpl
mpl.rcParams['figure.figsize'] = [8.0, 5.0]
mpl.rcParams['figure.dpi'] = 500
mpl.rcParams['savefig.dpi'] = 500

mpl.rcParams['font.size'] = 12
mpl.rcParams['legend.fontsize'] = 'large'
mpl.rcParams['figure.titlesize'] = 'medium'
mpl.rcParams['lines.linewidth']= 2.0

In [ ]:
# now it's big and pretty.

hycom_data.ssh.plot()

Note if you know you are going to plot the same bit of data over and over again to fiddle with the plot, you can download the data you need and save it in an array to make the proccess faster.

In [ ]:
SST = hycom_data_3D.water_temp[0,0,:,:]

In [ ]:
type(SST)

In [ ]:
SST # note it saves all the coordinates I need.

In [ ]:
SST.plot()

I can also save this subset of the data to a new netcdf file locally very easily. See http://xarray.pydata.org/en/stable/io.html for more details.

In [ ]:
SST.to_netcdf('SST_2001_001_01.nc') # the new netcdf file is saved in the local directory

And then I load in the new netcdf file in the same way as I did the remote data, but using the local filepath

In [ ]:
sst_data = xr.open_dataset('SST_2001_001_01.nc', decode_times=False)

In [ ]:
sst_data

# Lab 6.1

**E.0** Finish Lab 5.2 if you haven't already.

**E.1** Complete Introduction to Data Visualization with Matplotlib Chapters 1-2. Let me know if this feels like a good pace

**E.2** Make notes for yourself on progamming tecniques and commands you learned in the lecture and datacamp chapter above, including examples, comments and explainitory text. You can do this here or in a separate notebook that you link to here. Basically, you are making a cheat sheet for yourself.

See also http://xarray.pydata.org/en/stable/plotting.html for more info about plotting right from xarray (optional).

**E.3** Using the lecture as a guide, save the sea surface temperature at ~100 m depth on your birthday in 2019 as a new, local netcdf file. You don't have to submit the file, just the code here.

**E.4** Plot the above (your 2019 birthday SST) using the basic xarray funcitons

**E.5** What is the SST for Flower Garden Banks national marine sanctuary on your birthday? Show your work